# Prompt caching — analyse des 4 facteurs

Étude sur l'API Claude (Haiku 4.5), corpus = Constitution française 1958 (~20 800 tokens), 10 tours par configuration.

Toutes les valeurs proviennent du champ `usage` des réponses API. Aucune estimation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Tarifs Haiku 4.5, $/MTok — a reverifier sur la page officielle
PRICE = {"input": 1.00, "read": 0.10, "w5": 1.25, "w1": 2.00, "out": 5.00}

m = pd.read_csv("results/measures.csv", dtype={"config_id": str})
s = pd.read_csv("results/summary.csv", dtype={"config_id": str})

m["cost"] = (
    m.input_tokens * PRICE["input"]
    + m.cache_read_tokens * PRICE["read"]
    + m.cache_write_5m_tokens * PRICE["w5"]
    + m.cache_write_1h_tokens * PRICE["w1"]
    + m.output_tokens * PRICE["out"]
) / 1_000_000

sv = s.set_index("config_id").savings_pct
s[["config_id", "config_name", "total_cost_usd", "savings_pct"]]

## Facteur 1 — Variabilité du préfixe : effet dose-réponse

Configs 03 (extreme), 04 (moyen), 02 (stable). Le piège est un gradient, pas un interrupteur.

In [ ]:
for cid, lab in [("03", "extreme"), ("04", "moyen"), ("02", "stable")]:
    d = m[m.config_id == cid]
    reecritures = ((d.cache_write_5m_tokens + d.cache_write_1h_tokens) > 15000).sum()
    print(f"{lab:<8} -> {sv[cid]:6.1f} %   ({reecritures}/10 reecritures completes)")

## Facteur 2 — Performance vs robustesse (grille 3x3)

La metrique cle est l'**etendue** (max - min) par strategie : plus elle est faible, plus la strategie est robuste.

In [ ]:
grille = {
    "Automatique":          ["03", "04", "02"],
    "Breakpoint explicite": ["05", "06", "07"],
    "Melange TTL":          ["08", "09", "10"],
}
rows = []
for strat, ids in grille.items():
    vals = [sv[c] for c in ids]
    rows.append({"strategie": strat, "extreme": vals[0], "moyen": vals[1],
                 "stable": vals[2], "etendue_pts": max(vals) - min(vals)})
rob = pd.DataFrame(rows)
rob

Le breakpoint explicite atteint presque la performance de l'automatique (73,6 % vs 74,4 %) avec une etendue ~40x plus faible : c'est le meilleur compromis performance/robustesse.

## Facteur 3 — L'assurance asymétrique du TTL 1 h

Le meme dispositif (cache 1 h) aide contre l'expiration mais aggrave contre l'invalidation.

In [ ]:
print(f"Expiration   : simple {sv['13']:.1f}%  ->  mixte {sv['14']:.1f}%   ({sv['14']-sv['13']:+.1f} pts)  [assurance UTILE]")
print(f"Invalidation : simple {sv['11']:.1f}%  ->  mixte {sv['12']:.1f}%   ({sv['12']-sv['11']:+.1f} pts)  [assurance NEFASTE]")
print()
print("Cout du tour 4 (millieme de $) — le choc de la perturbation :")
for cid, lab in [("11", "invalidation simple"), ("12", "invalidation + TTL 1h"),
                 ("13", "expiration simple"), ("14", "expiration + TTL 1h")]:
    c4 = m[(m.config_id == cid) & (m.turn == 4)].cost.iloc[0] * 1000
    print(f"  {lab:<24} {c4:5.1f}")

## Facteur 4 — Le pré-chauffage optimise la variance

Economie identique, mais volatilite du cout par tour divisee par ~25.

In [ ]:
amp = s.set_index("config_id").amplitude
print(f"Economie      : config 15 (prewarm) = {sv['15']:.1f}%  |  config 10 (equiv) = {sv['10']:.1f}%  -> ecart non concluant")
print(f"Amplitude cout: config 15 = {amp['15']*1000:.1f}  |  config 10 = {amp['10']*1000:.1f} milliemes de $  -> facteur {amp['10']/amp['15']:.0f}x")
print()
print("=> le pre-chauffage n'achete pas du cout, il achete de la previsibilite.")

## Synthèse : matrice de décision

| Situation | Stratégie | Gain |
|---|---|---|
| Préfixe stable, contrôle total | Automatique | 74,4 % |
| Contenu variable en fin de prompt | Breakpoint explicite | ~72 %, insensible |
| Silences > 5 min | Mélange TTL | +4 pts |
| System modifié en cours | Éviter le TTL 1 h | −14,6 pts sinon |
| Latence / facturation à lisser | Pré-chauffage | volatilité ÷25 |

**Limites** : n=1 par config, un seul modèle, coûts calculés non facturés, tarifs datés du 24/07/2026.